In [ ]:
import torch
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
import time
print('cuda available:', torch.cuda.is_available())
torch.set_default_device(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

In [ ]:
# W = H = L = 768
WIDTH, HEIGHT = 70, 70
DIM = (1, 2, HEIGHT, WIDTH)  # format (N=batch_size, C=channel, H, W)

In [ ]:
# @title
def generateIdentity():
  '''
  Generate an identity flow field of dimension (N=1, HEIGHT, WIDTH, C=2)
  '''
  # permute into (N=1, height, width, C=2)
  affine_identity = torch.eye(3)[:2].expand((1, 2, 3))
  return F.affine_grid(affine_identity, DIM, align_corners=True).permute(0,3,1,2)

In [ ]:
# @title
def print_tensor(field):
  '''
  Print a tensor in a readable format.
  '''
  # get height, width
  h, w = field.size()[2:]
  for y in range(h):  # iterate row
    for x in range(w):  # iterate column
      print(f'({field[0][0][y][x].item():6.2f}, {field[0][1][y][x].item():6.2f})', end=', ')
    print()

In [ ]:
# @title
def plot_field(field, title='', **kwargs):
  fig, ax = plt.subplots()

  id = generateIdentity()
  plt.quiver(id[0][0], id[0][1], field[0][0], field[0][1], **kwargs)
  plt.title(title)

  # ax.invert_yaxis()  # does not work?
  plt.axis('square')
  plt.show()

In [ ]:
# @title
def max_norm(field, exclude : int = 0):
  '''
  Calculate the maximum norm of a tensor of dimension (N, C, H, W).
  '''
  return torch.max(torch.linalg.vector_norm(field, dim=(1,))).item()  # calculate norm over dimension 1 = C

In [ ]:
def remove_border(field, border_percentage):
  '''
  Zero out the border of a field for a given percentage of the size.
  '''
  h, w = field.size()[2:]
  bh, bw = int(h*border_percentage), int(w*border_percentage)

  ret = torch.zeros(field.size())
  ret[0, :, bh:-bh, bw:-bw] = field[0, :, bh:-bh, bw:-bw]

  return ret


In [ ]:
# @title
def timestepFD(flow_field):
  '''
  Calculate N so that max_norm(flow_field / 2**N) <= 0.5
  '''
  maxnorm = max_norm(flow_field)
  if maxnorm <= 0:
    return 0
  else:
    return math.ceil(max(math.log2(2*maxnorm), 1))

In [ ]:
def fastVectorFieldExponential(flow_field, N=None):
  '''
  Calculate the exponential of a vector field of dimension (N, C, H, W).
  '''
  # calculate N
  # print('Max norm:', max_norm(flow_field))
  if N is None:
    N = timestepFD(flow_field)
  dt = 2**(-N)
  # print(f'{N=}, {dt=}')

  # normalize flow_field
  h, w = flow_field.size()[2:]
  flow_field_copy = torch.clone(flow_field)
  flow_field_copy[0, 0, ...] /= w/2
  flow_field_copy[0, 1, ...] /= h/2

  identity = generateIdentity()
  # printu(u); print('-------------')
  v = identity + dt*flow_field_copy

  for _ in range(N):
    v_permuted = v.permute(0, 2, 3, 1)  # grid_sample grid argument needs (N, H, W, C)
    v = F.grid_sample(v, v_permuted, align_corners=True, padding_mode='border')
  v -= identity

  # denormalize
  # v[0][0] *= w/2
  # v[0][1] *= h/2
  return v

In [ ]:
def fastVectorFieldExponential_new(flow_field, N=None):
  '''
  Calculate the exponential of a vector field of dimension (N, C, H, W).
  '''
  # calculate N
  # print('Max norm:', max_norm(flow_field))
  if N is None:
    N = timestepFD(flow_field)
  dt = 2**(-N)
  # print(f'{N=}, {dt=}')

  # normalize flow_field
  h, w = flow_field.size()[2:]
  flow_field_copy = torch.clone(flow_field)
  flow_field_copy[0, 0, ...] /= w/2
  flow_field_copy[0, 1, ...] /= h/2

  identity = generateIdentity()
  # printu(u); print('-------------')
  v = dt*flow_field_copy

  for _ in range(N):
    v_permuted = (identity + v).permute(0, 2, 3, 1)  # grid_sample grid argument needs (N, H, W, C)
    v += F.grid_sample(v, v_permuted, align_corners=True, padding_mode='border')

  # denormalize
  # v[0][0] *= w/2
  # v[0][1] *= h/2
  return v

In [ ]:
def normalVectorFieldExponential_new(flow_field, N=None):
  '''
  Calculate the exponential of a vector field.
  '''
  if N is None:
    N = timestepFD(flow_field)
  dt = 2**(-N)

  # normalize u
  h, w = flow_field.size()[2:]
  flow_field_copy = torch.clone(flow_field)
  flow_field_copy[0, 0, ...] /= w/2
  flow_field_copy[0, 1, ...] /= h/2

  identity = generateIdentity()
  v = dt*flow_field_copy
  s = v

  for _ in range(2**N):
    s = F.grid_sample(v, (s + identity).permute(0, 2, 3, 1), align_corners=True, padding_mode='border') + s

  # denormalize
  # v[0][0] *= w/2
  # v[0][1] *= h/2
  return s

In [ ]:
def normalVectorFieldExponential(flow_field, N=None):
  '''
  Calculate the exponential of a vector field.
  '''
  if N is None:
    N = timestepFD(flow_field)
  dt = 2**(-N)

  # normalize u
  h, w = u.size()[2:]
  flow_field_copy = torch.clone(flow_field)
  flow_field_copy[0, 0, ...] /= w/2
  flow_field_copy[0, 1, ...] /= h/2

  identity = generateIdentity()

  v = identity + dt*flow_field_copy
  v_start = v.permute(0, 2, 3, 1)  # grid_sample grid argument needs (N, H, W, C)

  for _ in range(2**N):
    v = F.grid_sample(v, v_start, align_corners=True, padding_mode='border')
  v -= identity

  # denormalize
  # v[0][0] *= w/2
  # v[0][1] *= h/2
  return v



In [ ]:
def compareForwardBackward(flow_field, exp_function, N=None):
  start = time.time()

  # print()
  forward = exp_function(flow_field, N)
  backward = exp_function(-flow_field, N)

  id = generateIdentity()
  # plot_field(forward, title=exp_function.__name__, scale_units='xy', scale=1)

  residual = forward - F.grid_sample(backward, (forward+id).permute(0,2,3,1), align_corners=True, padding_mode='border')

  elapsed = int((time.time() - start)*1000)
  # print(f'Elapsed time {exp_function.__name__} for {N=}: {elapsed}ms')

  max_error = max_norm(remove_border(residual, 0.15))
  # plot_field(residual, title=exp_function.__name__, scale_units='xy', scale=1)
  return max_error, elapsed


In [ ]:
def plot_error(flow_field, exp_function):
  fig, ax = plt.subplots()

  N_default = timestepFD(flow_field)
  Ns = list(range(1, (2*N_default if exp_function.__name__.startswith('normal') else 5*N_default)))
  errs, times = zip(*(compareForwardBackward(flow_field, exp_function, N) for N in Ns))

  color = 'tab:red'
  # ax.set_xlabel('N')
  ax.set_ylabel('error (px)', color='tab:blue')
  ax.plot(Ns, errs, color='tab:blue', marker='o')
  ax.plot(N_default, compareForwardBackward(flow_field, exp_function, N_default)[0], 'go')
  # ax.ylim(bottom=0, top=max(errs)*1.4)
  ax.tick_params(axis='y', labelcolor='tab:blue')

  ax2 = ax.twinx()  # instantiate a second Axes that shares the same x-axis
  ax2.set_ylabel('time (ms)', color='tab:red')  # we already handled the x-label with ax1
  ax2.plot(Ns, times, color='tab:red')
  ax2.tick_params(axis='y', labelcolor='tab:red')

  ax.set_title(exp_function.__name__)
  fig.tight_layout()  # otherwise the right y-label is slightly clipped
  plt.show()


In [ ]:
# u = torch.ones(dim, dtype = torch.float)

## 1/2 rescaling
# u = -generateIdentity() / 2 * WIDTH/2

### rotate by angle
angle = math.pi/4
affine_matrix = torch.tensor([[[math.cos(angle), -math.sin(angle), 0],[math.sin(angle), math.cos(angle), 0]]], dtype=torch.float).expand((1, 2, 3))
affine_rotation = F.affine_grid(affine_matrix, DIM, align_corners=True).permute(0,3,1,2)
u = (affine_rotation - generateIdentity()) * WIDTH/2

# print_tensor(u)
# plot_field(u)
# plot_field(remove_border(u, 0.15))

fns = (fastVectorFieldExponential, fastVectorFieldExponential_new, normalVectorFieldExponential, normalVectorFieldExponential_new)
for fn in fns:
  # compareForwardBackward(u, fn)
  plot_error(u, fn)


In [ ]:
# @title
'''
forward_fast = fastVectorFieldExponential(u)
backward_fast = fastVectorFieldExponential(-u)

id = generateIdentity()
# print_tensor(forward)
plot_field(forward_fast-id, scale_units='xy', scale=1)
# plot_field(backward-id, scale_units='xy', scale=1)

# transformed = F.grid_sample(id, torch.permute(forward, (0,2,3,1))/10, align_corners=True, padding_mode='border')
transformed = F.grid_sample(forward_fast, (backward_fast-id).permute(0,2,3,1)/WIDTH*2, align_corners=True, padding_mode='border')
torch.allclose(id, transformed)
# plot_field(transformed-id, scale_units='xy', scale=1)
# print_tensor(transformed*width/2)
print('Max error norm:', max_norm(transformed*WIDTH/2))

# print(F.grid_sample(u, torch.permute(id, (0,2,3,1)), align_corners=True, padding_mode='border'))
# print_tensor(id-transformed)
print('Max error norm:', max_norm(id-transformed))
'''


In [ ]:
# @title
# forward_fast = fastVectorFieldExponential_new(u)
# backward_fast = fastVectorFieldExponential_new(-u)

# id = generateIdentity()
# print_tensor(forward)
# plot_field(forward_fast, scale_units='xy', scale=1)
# plot_field(backward-id, scale_units='xy', scale=1)

# transformed = F.grid_sample(id, torch.permute(forward, (0,2,3,1))/10, align_corners=True, padding_mode='border')
# transformed = F.grid_sample(forward_fast+id, backward_fast.permute(0,2,3,1)/WIDTH*2, align_corners=True, padding_mode='border')
# torch.allclose(id, transformed)
# plot_field(transformed-id, scale_units='xy', scale=1)
# print_tensor(transformed*width/2)
# print('Max error norm:', max_norm(transformed*WIDTH/2))

# print(F.grid_sample(u, torch.permute(id, (0,2,3,1)), align_corners=True, padding_mode='border'))
# print_tensor(id-transformed)
# print('Max error norm:', max_norm(id-transformed))

In [ ]:
# @title
# forward_normal = normalVectorFieldExponential(u)
# backward_normal = normalVectorFieldExponential(-u)

# plot_field(forward_normal-id, scale_units='xy', scale=1)

# transformed2 = F.grid_sample(forward_normal, (backward_normal-id).permute(0,2,3,1)/WIDTH*2, align_corners=True, padding_mode='border')
# print('Max error norm:', max_norm(transformed2*WIDTH/2))

